# Weekly Price Fluctuation Prediction Models

This notebook predicts **price fluctuations** (not absolute prices) one week ahead using three different models:
- K-Nearest Neighbors (KNN)
- Decision Tree
- Linear Regression

Unlike existing notebooks that predict the actual price value, this notebook focuses on predicting the **change in price** (fluctuation).

---

## Approach

1. Load and prepare data from `model_data.csv`
2. Create target variable: **price_fluctuation_1w** (price change from current week to next week)
3. Select relevant lagged features
4. Train-test split (80/20)
5. Train three models: KNN, Decision Tree, Linear Regression
6. Compare performance using RMSE, MAE, MAPE, R²
7. Analyze which model best predicts price fluctuations

---

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Machine learning libraries
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully!")

## Step 1: Load Data and Create Fluctuation Target

In [ ]:
# Load the prepared model data
df_model = pd.read_csv('data/processed/model_data.csv', parse_dates=['Date'], index_col='Date')

print(f"Loaded model data: {df_model.shape[0]} rows, {df_model.shape[1]} columns")
print(f"Date range: {df_model.index.min().date()} to {df_model.index.max().date()}")

# Create price fluctuation target
# Fluctuation = Future Price - Current Price
# We want to predict: price_1w_ahead - price_lag_1w (which is the current week's price shifted)
df_model['price_fluctuation_1w'] = df_model['price_1w_ahead'] - df_model['Europe_Base_Price']

# Also create percentage change for analysis
df_model['price_pct_fluctuation_1w'] = ((df_model['price_1w_ahead'] - df_model['Europe_Base_Price']) / df_model['Europe_Base_Price']) * 100

print("\n" + "="*80)
print("PRICE FLUCTUATION STATISTICS")
print("="*80)
print(f"\nFluctuation (absolute):")
print(df_model['price_fluctuation_1w'].describe())
print(f"\nFluctuation (percentage):")
print(df_model['price_pct_fluctuation_1w'].describe())

# Visualize fluctuation distribution
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].hist(df_model['price_fluctuation_1w'].dropna(), bins=50, edgecolor='black')
axes[0].set_title('Distribution of Weekly Price Fluctuation (Absolute)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Price Change (USD)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Change')
axes[0].legend()

axes[1].hist(df_model['price_pct_fluctuation_1w'].dropna(), bins=50, edgecolor='black', color='orange')
axes[1].set_title('Distribution of Weekly Price Fluctuation (Percentage)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Price Change (%)')
axes[1].set_ylabel('Frequency')
axes[1].axvline(x=0, color='red', linestyle='--', linewidth=2, label='No Change')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n✓ Target variable created: price_fluctuation_1w")

## Step 2: Prepare Features and Target

In [ ]:
# Drop rows with missing target
df_clean = df_model.dropna(subset=['price_fluctuation_1w']).copy()

print(f"Rows after removing missing targets: {len(df_clean)}")

# Select lagged features only (to avoid data leakage)
lag_features = [col for col in df_clean.columns if '_lag_' in col]

# Also include basic rolling statistics
feature_list = lag_features + [
    col for col in df_clean.columns 
    if any(x in col for x in ['_roll_', '_pct_change_', '_cv_']) and '_lag_' in col
]
feature_list = list(set(feature_list))  # Remove duplicates

print(f"\nTotal features available: {len(feature_list)}")

# Use top 30 features based on correlation with fluctuation
X_all = df_clean[feature_list]
y = df_clean['price_fluctuation_1w']

# Calculate correlations
correlations = X_all.corrwith(y).abs().sort_values(ascending=False)
top_30_features = correlations.head(30).index.tolist()

print(f"\nTop 30 Features (by correlation with fluctuation):")
for i, feat in enumerate(top_30_features[:10], 1):
    print(f"  {i:2d}. {feat:50s} (corr: {correlations[feat]:.4f})")
print("  ...")

# Use these features
X = df_clean[top_30_features]
y = df_clean['price_fluctuation_1w']

print(f"\n✓ Using {len(top_30_features)} features for modeling")

## Step 3: Train-Test Split

In [ ]:
# Time-based split (80/20)
split_idx = int(len(df_clean) * 0.8)
X_train = X.iloc[:split_idx]
X_test = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test = y.iloc[split_idx:]

print("="*80)
print("TRAIN-TEST SPLIT")
print("="*80)
print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {len(top_30_features)}")
print(f"\nTrain date range: {X_train.index.min().date()} to {X_train.index.max().date()}")
print(f"Test date range: {X_test.index.min().date()} to {X_test.index.max().date()}")

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Features scaled using StandardScaler")

## Step 4: Define Evaluation Function

In [ ]:
def evaluate_fluctuation_model(y_true, y_pred, model_name):
    """
    Evaluate model performance for price fluctuation prediction.
    """
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    
    # MAPE for fluctuation (avoid division by zero)
    # We'll use MAE / mean(abs(y_true)) as a percentage metric
    mean_abs_fluctuation = np.mean(np.abs(y_true))
    relative_mae = (mae / mean_abs_fluctuation) * 100 if mean_abs_fluctuation != 0 else 0
    
    r2 = r2_score(y_true, y_pred)
    
    # Direction accuracy (did we predict the right direction?)
    correct_direction = np.sum((y_true > 0) == (y_pred > 0))
    direction_accuracy = (correct_direction / len(y_true)) * 100
    
    metrics = {
        'Model': model_name,
        'RMSE': rmse,
        'MAE': mae,
        'Relative_MAE_%': relative_mae,
        'R²': r2,
        'Direction_Accuracy_%': direction_accuracy
    }
    
    print(f"\n{'='*70}")
    print(f"{model_name} Performance")
    print(f"{'='*70}")
    print(f"RMSE:                ${rmse:.2f}")
    print(f"MAE:                 ${mae:.2f}")
    print(f"Relative MAE:        {relative_mae:.2f}%")
    print(f"R²:                  {r2:.4f}")
    print(f"Direction Accuracy:  {direction_accuracy:.2f}%")
    print(f"{'='*70}")
    
    return metrics

# Store results
results = []

print("Evaluation function defined!")

## Step 5: Model 1 - Linear Regression

In [ ]:
print("\n" + "="*80)
print("MODEL 1: LINEAR REGRESSION")
print("="*80)

# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
metrics_lr = evaluate_fluctuation_model(y_test, y_pred_lr, 'Linear Regression')
results.append(metrics_lr)

print("\n✓ Linear Regression trained and evaluated")

## Step 6: Model 2 - Decision Tree

In [ ]:
print("\n" + "="*80)
print("MODEL 2: DECISION TREE with GridSearchCV")
print("="*80)

# Define parameter grid
param_grid_dt = {
    'max_depth': [5, 10, 15, 20],
    'min_samples_split': [5, 10, 20],
    'min_samples_leaf': [2, 5, 10]
}

# Grid search
dt_base = DecisionTreeRegressor(random_state=42)
grid_dt = GridSearchCV(dt_base, param_grid_dt, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

print("\nTraining Decision Tree with GridSearchCV...")
grid_dt.fit(X_train_scaled, y_train)

# Best model
dt_model = grid_dt.best_estimator_
print(f"\nBest parameters: {grid_dt.best_params_}")

# Predict
y_pred_dt = dt_model.predict(X_test_scaled)

# Evaluate
metrics_dt = evaluate_fluctuation_model(y_test, y_pred_dt, 'Decision Tree')
results.append(metrics_dt)

print("\n✓ Decision Tree trained and evaluated")

## Step 7: Model 3 - K-Nearest Neighbors (KNN)

In [ ]:
print("\n" + "="*80)
print("MODEL 3: K-NEAREST NEIGHBORS with GridSearchCV")
print("="*80)

# Define parameter grid
param_grid_knn = {
    'n_neighbors': [3, 5, 7, 10, 15],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

# Grid search
knn_base = KNeighborsRegressor()
grid_knn = GridSearchCV(knn_base, param_grid_knn, cv=5, scoring='neg_mean_squared_error', n_jobs=-1, verbose=1)

print("\nTraining KNN with GridSearchCV...")
grid_knn.fit(X_train_scaled, y_train)

# Best model
knn_model = grid_knn.best_estimator_
print(f"\nBest parameters: {grid_knn.best_params_}")

# Predict
y_pred_knn = knn_model.predict(X_test_scaled)

# Evaluate
metrics_knn = evaluate_fluctuation_model(y_test, y_pred_knn, 'K-Nearest Neighbors')
results.append(metrics_knn)

print("\n✓ KNN trained and evaluated")

## Step 8: Model Comparison

In [ ]:
print("\n" + "="*80)
print("MODEL COMPARISON - PRICE FLUCTUATION PREDICTION")
print("="*80)

# Create comparison dataframe
results_df = pd.DataFrame(results)
print("\n", results_df.to_string(index=False))

# Determine best model
best_model_idx = results_df['RMSE'].idxmin()
best_model_name = results_df.loc[best_model_idx, 'Model']

print(f"\n{'='*80}")
print(f"BEST MODEL: {best_model_name}")
print(f"{'='*80}")
print(f"RMSE: ${results_df.loc[best_model_idx, 'RMSE']:.2f}")
print(f"Direction Accuracy: {results_df.loc[best_model_idx, 'Direction_Accuracy_%']:.2f}%")
print(f"{'='*80}")

# Visualize comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# RMSE comparison
axes[0].bar(results_df['Model'], results_df['RMSE'], color=['blue', 'green', 'orange'])
axes[0].set_title('RMSE Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('RMSE ($)')
axes[0].tick_params(axis='x', rotation=45)

# MAE comparison
axes[1].bar(results_df['Model'], results_df['MAE'], color=['blue', 'green', 'orange'])
axes[1].set_title('MAE Comparison', fontsize=14, fontweight='bold')
axes[1].set_ylabel('MAE ($)')
axes[1].tick_params(axis='x', rotation=45)

# Direction Accuracy comparison
axes[2].bar(results_df['Model'], results_df['Direction_Accuracy_%'], color=['blue', 'green', 'orange'])
axes[2].set_title('Direction Accuracy Comparison', fontsize=14, fontweight='bold')
axes[2].set_ylabel('Direction Accuracy (%)')
axes[2].axhline(y=50, color='red', linestyle='--', linewidth=1, label='Random Guess (50%)')
axes[2].legend()
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Save results
results_df.to_csv('data/processed/weekly_fluctuation_model_comparison.csv', index=False)
print("\n✓ Results saved to data/processed/weekly_fluctuation_model_comparison.csv")

## Step 9: Prediction Visualization

In [ ]:
# Create prediction dataframe
pred_df = pd.DataFrame({
    'Date': y_test.index,
    'Actual_Fluctuation': y_test.values,
    'LR_Predicted': y_pred_lr,
    'DT_Predicted': y_pred_dt,
    'KNN_Predicted': y_pred_knn
}).set_index('Date')

# Plot predictions
fig = go.Figure()

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['Actual_Fluctuation'], 
                         mode='lines+markers', name='Actual Fluctuation',
                         line=dict(color='black', width=2)))

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['LR_Predicted'],
                         mode='lines', name='Linear Regression',
                         line=dict(color='blue', dash='dash')))

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['DT_Predicted'],
                         mode='lines', name='Decision Tree',
                         line=dict(color='green', dash='dash')))

fig.add_trace(go.Scatter(x=pred_df.index, y=pred_df['KNN_Predicted'],
                         mode='lines', name='KNN',
                         line=dict(color='orange', dash='dash')))

# Add zero line
fig.add_hline(y=0, line_dash="dot", line_color="red", opacity=0.5,
              annotation_text="No Change", annotation_position="right")

fig.update_layout(
    title='Actual vs Predicted Weekly Price Fluctuations (Test Set)',
    xaxis_title='Date',
    yaxis_title='Price Fluctuation (USD)',
    height=600,
    hovermode='x unified',
    legend=dict(x=0.01, y=0.99)
)

fig.show()

# Save predictions
pred_df.to_csv('data/processed/weekly_fluctuation_predictions.csv')
print("\n✓ Predictions saved to data/processed/weekly_fluctuation_predictions.csv")

## Step 10: Direction Prediction Analysis

In [ ]:
print("\n" + "="*80)
print("DIRECTION PREDICTION ANALYSIS")
print("="*80)

# Analyze direction predictions for each model
pred_df['Actual_Direction'] = (pred_df['Actual_Fluctuation'] > 0).astype(int)
pred_df['LR_Direction'] = (pred_df['LR_Predicted'] > 0).astype(int)
pred_df['DT_Direction'] = (pred_df['DT_Predicted'] > 0).astype(int)
pred_df['KNN_Direction'] = (pred_df['KNN_Predicted'] > 0).astype(int)

# Count correct directions
lr_correct = (pred_df['Actual_Direction'] == pred_df['LR_Direction']).sum()
dt_correct = (pred_df['Actual_Direction'] == pred_df['DT_Direction']).sum()
knn_correct = (pred_df['Actual_Direction'] == pred_df['KNN_Direction']).sum()

total = len(pred_df)

print(f"\nLinear Regression: {lr_correct}/{total} correct ({(lr_correct/total)*100:.2f}%)")
print(f"Decision Tree:     {dt_correct}/{total} correct ({(dt_correct/total)*100:.2f}%)")
print(f"KNN:               {knn_correct}/{total} correct ({(knn_correct/total)*100:.2f}%)")

# Confusion matrix for best model
from sklearn.metrics import confusion_matrix, classification_report

print(f"\n" + "="*80)
print(f"CONFUSION MATRIX - {best_model_name}")
print("="*80)

if best_model_name == 'Linear Regression':
    cm = confusion_matrix(pred_df['Actual_Direction'], pred_df['LR_Direction'])
elif best_model_name == 'Decision Tree':
    cm = confusion_matrix(pred_df['Actual_Direction'], pred_df['DT_Direction'])
else:
    cm = confusion_matrix(pred_df['Actual_Direction'], pred_df['KNN_Direction'])

print("\nPredicted:    Down (0)  Up (1)")
print(f"Actual Down:  {cm[0][0]:8d}  {cm[0][1]:6d}")
print(f"Actual Up:    {cm[1][0]:8d}  {cm[1][1]:6d}")

print("\n" + "="*80)

## Summary

This notebook successfully trained and compared three models for predicting **weekly price fluctuations**:

1. **Linear Regression**: Simple and interpretable
2. **Decision Tree**: Captures non-linear relationships
3. **K-Nearest Neighbors**: Instance-based learning

Key metrics evaluated:
- **RMSE & MAE**: Measure prediction error magnitude
- **Direction Accuracy**: How often the model correctly predicts if prices will go up or down
- **R²**: Proportion of variance explained

The best performing model is selected based on RMSE, and all results are saved for further analysis.